# MOVIE RECOMMENDER SYSTEM

## Project Introduction
This project aims to develop a machine learning model that predicts movie ratings based on user–movie interactions and movie-related features. The goal is to generate accurate rating predictions for unseen user–movie pairs using the provided training and test datasets.
The project implements a hybrid recommendation approach that combines two models: Singular Value Decomposition (SVD), which uses collaborative filtering to learn patterns from historical user ratings, and LightGBM, which predicts ratings using engineered movie features derived from tag-genome relevance scores and available IMDb metadata.
The models are evaluated using Root Mean Squared Error (RMSE), where lower values indicate better prediction accuracy. A validation dataset is used to compare model performance and determine the optimal weight for combining their predictions. Finally, both models are retrained using the full training dataset, and their predictions are combined to generate the final submission.csv file.

## Project Workflow
The project is organized into the following major steps:

1.	Environment Setup and Data Loading

3.	Data Splitting and Baseline Modeling 
4.	Movie Feature Engineering
5.	LightGBM Model Training 
6.	Ensemble Optimization 
7.	Final Model Training 
8.	Prediction and Submission Generation 

## Step 1: Set up the environment and load the data

In [1]:
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from surprise import Dataset, Reader, SVD
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

# Suppress non-critical warnings for cleaner notebook output.
warnings.filterwarnings("ignore")

# Set a fixed random seed for reproducible results.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# Loading data
train = pd.read_csv(
    "train.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "rating": "float32"
    }
)

test = pd.read_csv(
    "test.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32"
    }
)

In [3]:
print("Training shape:", train.shape)
print("Test shape:", test.shape)

Training shape: (10000038, 4)
Test shape: (5000019, 2)


In [4]:
display(train.head())
display(test.head())

,userId,movieId,rating,timestamp
0,5163,57669,4.0,1518349992
1,106343,5,4.5,1206238739
2,146790,5459,5.0,1076215539
3,106362,32296,2.0,1423042565
4,9041,366,3.0,833375837


,userId,movieId
0,1,2011
1,1,4144
2,1,5767
3,1,6711
4,1,7318


In [5]:
# Check for missing values in the training data.
print("\nMissing values in training data:")
print(train.isnull().sum())


Missing values in training data:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [6]:
# Examine the distribution of known ratings.
print("\nRating statistics:")
print(train["rating"].describe())


Rating statistics:
count    1.000004e+07
mean     3.533395e+00
std      1.036790e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64


## Step 2: Split the data and train the baseline model

In [7]:
# Splitting the data
train_part, val_part = train_test_split(
    train,
    test_size=0.10,
    random_state=RANDOM_STATE
)

# Reset indices for easier handling.
train_part = train_part.reset_index(drop=True)
val_part = val_part.reset_index(drop=True)

print("Training rows:", len(train_part))
print("Validation rows:", len(val_part))

Training rows: 9000034
Validation rows: 1000004


In [ ]:
reader = Reader(rating_scale=(0.5, 5.0))

# Candidate configurations to test
configs = [
    (50, 30, 0.005, 0.05),
    (100, 40, 0.005, 0.05),
    (150, 40, 0.005, 0.05),
    (200, 40, 0.005, 0.05),
    (100, 50, 0.005, 0.02),
    (150, 50, 0.005, 0.02),
    (200, 50, 0.003, 0.05),
    (150, 40, 0.003, 0.10),
]

# Prepare the training dataset once
data = Dataset.load_from_df(
    train_part[["userId", "movieId", "rating"]],
    reader
)
full_trainset = data.build_full_trainset()

# Track the best configuration
results = []
best_rmse = float("inf")
best_svd = None
best_params = None
best_predictions = None

# Train and evaluate each configuration
for factors, epochs, lr, reg in configs:

    model = SVD(
        n_factors=factors,
        n_epochs=epochs,
        lr_all=lr,
        reg_all=reg,
        random_state=RANDOM_STATE
    )

    model.fit(full_trainset)

    predictions = np.array([
        model.predict(int(user), int(movie)).est
        for user, movie in zip(
            val_part["userId"],
            val_part["movieId"]
        )
    ])
    rmse = np.sqrt(mean_squared_error(
    val_part["rating"].values,
    predictions
    ))

    results.append({
        "n_factors": factors,
        "n_epochs": epochs,
        "learning_rate": lr,
        "regularization": reg,
        "RMSE": rmse
    })

    print(
        f"Factors={factors}, Epochs={epochs}, "
        f"LR={lr}, Reg={reg} -> RMSE={rmse:.6f}"
    )

    if rmse < best_rmse:
        best_rmse = rmse
        best_svd = model
        best_predictions = predictions
        best_params = {
            "n_factors": factors,
            "n_epochs": epochs,
            "lr_all": lr,
            "reg_all": reg
        }

# Display results, best first
results_df = pd.DataFrame(results).sort_values("RMSE")

print("\nBest SVD parameters:", best_params)
print(f"Best validation RMSE: {best_rmse:.6f}")

display(results_df)

# Keep these variables for the next notebook steps
svd_model = best_svd
svd_predictions = best_predictions
svd_rmse = best_rmse

Factors=50, Epochs=30, LR=0.005, Reg=0.05 -> RMSE=0.824702
Factors=100, Epochs=40, LR=0.005, Reg=0.05 -> RMSE=0.813309
